# QM640 Food Price Affordability AI
## Notebook 02 - Synopsis-Aligned Quality, Cleaning, and Target Engineering

**Student:** Piyush Soni  
**Environment:** Google Colab  
**Required predecessor:** Notebook 01 synopsis-aligned version

This notebook preserves raw values, performs exact calendar joins, creates the
original one-, two-, and three-month future price targets, and conditionally
integrates official confirmatory data without silently substituting proxies.


## 1. Runtime and persistent-storage setup

Notebook 01 saved its outputs in `MyDrive/QM640_Food_Affordability`. This notebook
uses the same location so files survive Colab restarts. pandas and NumPy are
already installed in Colab, avoiding unnecessary installation time.


In [1]:
from pathlib import Path
import json
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SAVE_TO_DRIVE = True

if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/QM640_Food_Affordability")
else:
    # The override supports local validation without writing test outputs into Git.
    default_root = Path.cwd().resolve()
    if default_root.name == "notebooks":
        default_root = default_root.parent
    OUTPUT_ROOT = Path(os.environ.get("QM640_OUTPUT_ROOT", str(default_root)))

PROCESSED = OUTPUT_ROOT / "data" / "processed"
REPORT_OUTPUT = OUTPUT_ROOT / "reports" / "notebook_outputs"
PROCESSED.mkdir(parents=True, exist_ok=True)
REPORT_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Running in Google Colab: {IN_COLAB}")
print(f"Input/output root: {OUTPUT_ROOT}")
print(f"pandas: {pd.__version__}; NumPy: {np.__version__}")


Mounted at /content/drive
Running in Google Colab: True
Input/output root: /content/drive/MyDrive/QM640_Food_Affordability
pandas: 2.2.2; NumPy: 2.0.2


## 2. Load and verify Notebook 01 outputs

Only the columns required for cleaning are loaded. The state file is already a
compact monthly aggregate, so it is safe to load in memory. Explicit prerequisite
checks produce a helpful error instead of a later, ambiguous failure.


In [2]:
STATE_INPUT = PROCESSED / "modeling_state_monthly.csv.gz"
NATIONAL_INPUT = PROCESSED / "modeling_monthly_national.csv.gz"
NB01_SUMMARY = REPORT_OUTPUT / "01_execution_summary.json"

required_files = [STATE_INPUT, NATIONAL_INPUT, NB01_SUMMARY]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Notebook 01 outputs are missing. Run Notebook 01 completely first.\n"
        + "\n".join(missing_files)
    )

state = pd.read_csv(
    STATE_INPUT,
    parse_dates=["date"],
    dtype={"region": "category", "Commodity": "category"},
    low_memory=False,
)
national = pd.read_csv(NATIONAL_INPUT, parse_dates=["date"], low_memory=False)
nb01_summary = json.loads(NB01_SUMMARY.read_text(encoding="utf-8"))

print("Notebook 01 status:", nb01_summary.get("status"))
print(f"State panel rows loaded: {len(state):,}")
print(f"National panel rows loaded: {len(national):,}")
print(f"State date range: {state['date'].min().date()} to {state['date'].max().date()}")
print(f"National date range: {national['date'].min().date()} to {national['date'].max().date()}")


Notebook 01 status: completed_with_confirmatory_data_gaps
State panel rows loaded: 41,792
National panel rows loaded: 319
State date range: 2001-01-01 to 2026-07-01
National date range: 2000-01-01 to 2026-07-01


## 3. Structural integrity checks

The intended state-panel key is `region + date + Commodity`. Duplicates would
distort statistics and train/test evaluation, so the notebook stops rather than
silently deleting them. Dates must be monthly timestamps at the first day.


In [3]:
state_key = ["region", "date", "Commodity"]
duplicate_keys = int(state.duplicated(state_key).sum())
national_duplicate_dates = int(national.duplicated(["date"]).sum())
non_month_start_state = int(state["date"].dt.day.ne(1).sum())
non_month_start_national = int(national["date"].dt.day.ne(1).sum())

structural_checks = {
    "state_key_duplicates": duplicate_keys,
    "national_date_duplicates": national_duplicate_dates,
    "state_dates_not_month_start": non_month_start_state,
    "national_dates_not_month_start": non_month_start_national,
}
print(pd.Series(structural_checks).to_string())

if any(structural_checks.values()):
    raise AssertionError(
        "Structural checks failed. Do not continue until duplicate/date problems are resolved."
    )


state_key_duplicates              0
national_date_duplicates          0
state_dates_not_month_start       0
national_dates_not_month_start    0


## 4. Missingness and reporting-coverage profile

Missingness is reported both overall and by commodity. Coverage counts are kept
because an unbalanced official reporting panel is not equivalent to random
missing data. The analysis does not replace absent prices with zero or interpolate
them across reporting gaps.


In [4]:
state_missingness = (
    state.isna()
    .mean()
    .mul(100)
    .rename("missing_percent")
    .rename_axis("variable")
    .reset_index()
    .sort_values("missing_percent", ascending=False)
)
national_missingness = (
    national.isna()
    .mean()
    .mul(100)
    .rename("missing_percent")
    .rename_axis("variable")
    .reset_index()
    .sort_values("missing_percent", ascending=False)
)

commodity_coverage = (
    state.groupby("Commodity", observed=True, as_index=False)
    .agg(
        rows=("date", "size"),
        states_uts=("region", "nunique"),
        start_date=("date", "min"),
        end_date=("date", "max"),
        official_observations=("source_rows", "sum"),
        missing_rainfall_pct=("rainfall_mm", lambda x: x.isna().mean() * 100),
        missing_temperature_pct=("temperature_c", lambda x: x.isna().mean() * 100),
    )
    .sort_values("official_observations", ascending=False)
)

state_missingness.to_csv(REPORT_OUTPUT / "02_state_missingness.csv", index=False)
national_missingness.to_csv(REPORT_OUTPUT / "02_national_missingness.csv", index=False)
commodity_coverage.to_csv(REPORT_OUTPUT / "02_commodity_coverage.csv", index=False)

print("Highest state-panel missingness")
print(state_missingness.head(12).round(3).to_string(index=False))
print("\nCommodity reporting coverage")
print(commodity_coverage.round(3).to_string(index=False))


Highest state-panel missingness
                  variable  missing_percent
                      date              0.0
                    region              0.0
                 Commodity              0.0
               source_rows              0.0
             daily_records              0.0
modal_price_rs_per_quintal              0.0
                admin_type              0.0
        reference_location              0.0
    spatial_representation              0.0
               rainfall_mm              0.0
             temperature_c              0.0
         temperature_max_c              0.0

Commodity reporting coverage
                  Commodity  rows  states_uts start_date   end_date  official_observations  missing_rainfall_pct  missing_temperature_pct
                      Wheat  4853          27 2001-04-01 2026-07-01                3470626                   0.0                      0.0
                     Potato  7297          32 2001-01-01 2026-07-01                3406949

## 5. Economic and physical validity rules

The rules are deliberately broad:

- mandi price and observation weights must be positive;
- rainfall cannot be negative;
- temperature must lie between −20°C and 60°C; and
- relative humidity must lie between 0% and 100%.

Broad bounds remove impossible values while avoiding the deletion of legitimate
extreme weather or price shocks—the phenomena the assignment aims to study.


In [5]:
validity_flags = pd.DataFrame(index=state.index)
validity_flags["invalid_price"] = (
    state["modal_price_rs_per_quintal"].isna()
    | state["modal_price_rs_per_quintal"].le(0)
)
validity_flags["invalid_source_rows"] = (
    state["source_rows"].isna() | state["source_rows"].le(0)
)
validity_flags["invalid_rainfall"] = (
    state["rainfall_mm"].notna() & state["rainfall_mm"].lt(0)
)
validity_flags["invalid_temperature"] = (
    state["temperature_c"].notna()
    & ~state["temperature_c"].between(-20, 60)
)
validity_flags["invalid_humidity"] = (
    state["relative_humidity_pct"].notna()
    & ~state["relative_humidity_pct"].between(0, 100)
)
validity_flags["any_invalid"] = validity_flags.any(axis=1)

validity_counts = (
    validity_flags.sum()
    .astype(int)
    .rename("rows")
    .rename_axis("rule")
    .reset_index()
)
validity_counts.to_csv(REPORT_OUTPUT / "02_validity_rule_counts.csv", index=False)
print(validity_counts.to_string(index=False))

# Keep a copy of all valid records; removed counts remain documented.
clean = state.loc[~validity_flags["any_invalid"]].copy()
print(f"Rows retained after validity rules: {len(clean):,} ({len(clean)/len(state):.2%})")


               rule  rows
      invalid_price     0
invalid_source_rows     0
   invalid_rainfall     0
invalid_temperature     0
   invalid_humidity     0
        any_invalid     0
Rows retained after validity rules: 41,792 (100.00%)


## 6. Conservative extreme-price treatment

Price distributions differ greatly by commodity. Outliers are therefore assessed
within commodity on the log-price scale using the robust median absolute
deviation (MAD). Only observations beyond six robust standard deviations are
flagged.

For model stability, flagged values receive a winsorized companion value bounded
by the commodity's 0.5th and 99.5th percentiles. The original price remains
unchanged in `modal_price_rs_per_quintal`, and `price_extreme_flag` makes every
adjustment auditable.


In [6]:
def robust_zscore(series: pd.Series) -> pd.Series:
    """Median/MAD z-score; fallback to standard deviation if MAD is zero."""
    median = series.median()
    mad = (series - median).abs().median()
    robust_scale = 1.4826 * mad
    if not np.isfinite(robust_scale) or robust_scale == 0:
        robust_scale = series.std()
    if not np.isfinite(robust_scale) or robust_scale == 0:
        robust_scale = 1.0
    return (series - median) / robust_scale


clean["log_price_raw"] = np.log(clean["modal_price_rs_per_quintal"])
clean["price_robust_z"] = (
    clean.groupby("Commodity", observed=True)["log_price_raw"]
    .transform(robust_zscore)
)
clean["price_extreme_flag"] = clean["price_robust_z"].abs().gt(6)

commodity_bounds = (
    clean.groupby("Commodity", observed=True)["modal_price_rs_per_quintal"]
    .quantile([0.005, 0.995])
    .unstack()
    .rename(columns={0.005: "price_lower_bound", 0.995: "price_upper_bound"})
)
clean = clean.join(commodity_bounds, on="Commodity")

# Only flagged observations are adjusted. Non-flagged prices remain exactly raw.
clean["price_cleaned_rs_per_quintal"] = clean["modal_price_rs_per_quintal"]
flagged = clean["price_extreme_flag"]
clean.loc[flagged, "price_cleaned_rs_per_quintal"] = clean.loc[
    flagged, "modal_price_rs_per_quintal"
].clip(
    lower=clean.loc[flagged, "price_lower_bound"],
    upper=clean.loc[flagged, "price_upper_bound"],
)
clean["log_price_cleaned"] = np.log(clean["price_cleaned_rs_per_quintal"])

outlier_summary = (
    clean.groupby("Commodity", observed=True, as_index=False)
    .agg(
        rows=("date", "size"),
        extreme_prices=("price_extreme_flag", "sum"),
        raw_min_price=("modal_price_rs_per_quintal", "min"),
        raw_max_price=("modal_price_rs_per_quintal", "max"),
        cleaned_min_price=("price_cleaned_rs_per_quintal", "min"),
        cleaned_max_price=("price_cleaned_rs_per_quintal", "max"),
    )
)
outlier_summary["extreme_percent"] = (
    100 * outlier_summary["extreme_prices"] / outlier_summary["rows"]
)
outlier_summary.to_csv(REPORT_OUTPUT / "02_extreme_price_summary.csv", index=False)
print(outlier_summary.round(3).to_string(index=False))


                  Commodity  rows  extreme_prices  raw_min_price  raw_max_price  cleaned_min_price  cleaned_max_price  extreme_percent
Arhar (Tur/Red Gram)(Whole)  3954               8         24.250      33877.714            350.000          33877.714            0.202
   Bengal Gram(Gram)(Whole)  4445               1         24.900      12780.000            145.000          12780.000            0.022
                      Onion  7039               1          2.500      10921.090             44.608          10921.090            0.014
                     Potato  7297               1          6.750      15257.666             49.575          15257.666            0.014
                       Rice  4784               3         19.333     136673.333            198.000          25700.667            0.063
                   Soyabean  2783               0        865.000      14666.667            865.000          14666.667            0.000
                     Tomato  6637               1      

## 7. Exact calendar-lag price features

`groupby().pct_change(12)` means twelve *rows*, not necessarily twelve calendar
months. In an unbalanced reporting panel that can incorrectly compare dates more
than a year apart.

The code below performs keyed self-merges for exactly one month and twelve months
earlier. If that calendar observation is absent, the change correctly remains
missing. This prevents both timing errors and future leakage.


In [7]:
clean = clean.sort_values(state_key).reset_index(drop=True)
lag_lookup = clean[
    ["region", "Commodity", "date", "price_cleaned_rs_per_quintal"]
].copy()


def add_exact_calendar_lag(
    frame: pd.DataFrame, lookup: pd.DataFrame, months: int
) -> pd.DataFrame:
    """Join the price from exactly `months` calendar months earlier."""
    lagged = lookup.copy()
    lagged["date"] = lagged["date"] + pd.DateOffset(months=months)
    lagged = lagged.rename(
        columns={
            "price_cleaned_rs_per_quintal": f"price_lag_{months}m"
        }
    )
    return frame.merge(
        lagged,
        on=["region", "Commodity", "date"],
        how="left",
        validate="one_to_one",
    )


clean = add_exact_calendar_lag(clean, lag_lookup, months=1)
clean = add_exact_calendar_lag(clean, lag_lookup, months=12)
clean["price_mom_pct"] = (
    clean["price_cleaned_rs_per_quintal"] / clean["price_lag_1m"] - 1
) * 100
clean["price_yoy_pct"] = (
    clean["price_cleaned_rs_per_quintal"] / clean["price_lag_12m"] - 1
) * 100
clean["has_exact_1m_lag"] = clean["price_lag_1m"].notna()
clean["has_exact_12m_lag"] = clean["price_lag_12m"].notna()

print(f"Rows with exact one-month lag: {clean['has_exact_1m_lag'].sum():,} ({clean['has_exact_1m_lag'].mean():.2%})")
print(f"Rows with exact twelve-month lag: {clean['has_exact_12m_lag'].sum():,} ({clean['has_exact_12m_lag'].mean():.2%})")


Rows with exact one-month lag: 40,189 (96.16%)
Rows with exact twelve-month lag: 37,090 (88.75%)


## 8. State-specific seasonal climate anomalies

Climate values repeat once for every reported commodity in the state price panel.
Using those repeated rows would over-weight months with more commodity reporting.
We first deduplicate to one climate record per state-month, estimate each state's
monthly climatology using completed years through 2025, and then merge anomalies
back to the price panel.


In [8]:
climate_columns = [
    "region",
    "date",
    "rainfall_mm",
    "temperature_c",
    "relative_humidity_pct",
]
state_climate = (
    clean[climate_columns]
    .drop_duplicates(["region", "date"])
    .copy()
)
state_climate["calendar_month"] = state_climate["date"].dt.month

# Exclude partial 2026 from the reference climatology.
reference_climate = state_climate.loc[state_climate["date"].dt.year <= 2025]
climatology = (
    reference_climate.groupby(
        ["region", "calendar_month"], observed=True, as_index=False
    )
    .agg(
        normal_rainfall_mm=("rainfall_mm", "mean"),
        normal_temperature_c=("temperature_c", "mean"),
        normal_humidity_pct=("relative_humidity_pct", "mean"),
    )
)
state_climate = state_climate.merge(
    climatology,
    on=["region", "calendar_month"],
    how="left",
    validate="many_to_one",
)
state_climate["rainfall_anomaly_mm"] = (
    state_climate["rainfall_mm"] - state_climate["normal_rainfall_mm"]
)
state_climate["rainfall_anomaly_pct"] = (
    100
    * state_climate["rainfall_anomaly_mm"]
    / state_climate["normal_rainfall_mm"].replace(0, np.nan)
)
state_climate["temperature_anomaly_c"] = (
    state_climate["temperature_c"]
    - state_climate["normal_temperature_c"]
)

anomaly_columns = [
    "region",
    "date",
    "rainfall_anomaly_mm",
    "rainfall_anomaly_pct",
    "temperature_anomaly_c",
]
clean = clean.merge(
    state_climate[anomaly_columns],
    on=["region", "date"],
    how="left",
    validate="many_to_one",
)
print(f"Rows with rainfall anomaly: {clean['rainfall_anomaly_pct'].notna().sum():,}")
print(f"Rows with temperature anomaly: {clean['temperature_anomaly_c'].notna().sum():,}")


Rows with rainfall anomaly: 41,790
Rows with temperature anomaly: 41,792


## 9. Clean the national monthly panel

National outcomes are retained as missing when the source does not report them.
No backward fill or interpolation is used because either can leak later
information into earlier dates. Exact date shifts create CPI lags.


In [9]:
national_clean = national.sort_values("date").copy()
national_clean["is_partial_2026"] = national_clean["date"].dt.year.eq(2026)

# Replace infinite percentage changes, which can arise from a zero denominator,
# with missing values rather than large artificial numbers.
national_clean = national_clean.replace([np.inf, -np.inf], np.nan)

cpi_lookup = national_clean[["date", "food_cpi_2015_100"]].copy()
for months in [1, 3, 6, 12]:
    lagged = cpi_lookup.copy()
    lagged["date"] = lagged["date"] + pd.DateOffset(months=months)
    lagged = lagged.rename(
        columns={"food_cpi_2015_100": f"food_cpi_lag_{months}m_exact"}
    )
    national_clean = national_clean.merge(
        lagged, on="date", how="left", validate="one_to_one"
    )

# Recalculate exact-calendar changes from the level series for consistency.
national_clean["food_cpi_mom_pct_exact"] = (
    national_clean["food_cpi_2015_100"]
    / national_clean["food_cpi_lag_1m_exact"]
    - 1
) * 100
national_clean["food_cpi_yoy_pct_exact"] = (
    national_clean["food_cpi_2015_100"]
    / national_clean["food_cpi_lag_12m_exact"]
    - 1
) * 100

national_validity = {
    "nonpositive_cpi": int(
        (
            national_clean["food_cpi_2015_100"].notna()
            & national_clean["food_cpi_2015_100"].le(0)
        ).sum()
    ),
    "negative_rainfall": int(
        (
            national_clean["state_avg_rainfall_mm"].notna()
            & national_clean["state_avg_rainfall_mm"].lt(0)
        ).sum()
    ),
    "temperature_outside_bounds": int(
        (
            national_clean["state_avg_temperature_c"].notna()
            & ~national_clean["state_avg_temperature_c"].between(-20, 60)
        ).sum()
    ),
}
print(pd.Series(national_validity).to_string())
if any(national_validity.values()):
    raise AssertionError("National validity checks failed.")


nonpositive_cpi               0
negative_rainfall             0
temperature_outside_bounds    0


## 10. Original-synopsis targets and optional confirmatory-source integration

The synopsis defines future essential-food price change at one-, two-, and
three-month horizons. These targets must be created by exact keyed calendar
joins, not by row offsets. This section also conditionally integrates
standardized arrivals, state crop production/yield, and rural wages when the
official files requested by Notebook 01 are present.

The following rules prevent conceptual drift:

- reporting counts remain coverage variables and are never renamed as arrivals;
- future targets use the state-commodity price panel, while national CPI remains
  a supplementary benchmark;
- unavailable confirmatory predictors remain explicitly missing;
- no missing official value is synthetically generated or silently zero-filled.

The saved confirmatory target columns are
`future_price_change_h1m_pct`, `future_price_change_h2m_pct`, and
`future_price_change_h3m_pct`.


In [10]:
EXTERNAL_REQUIRED = OUTPUT_ROOT / "data" / "external_required"
EXTERNAL_REQUIRED.mkdir(parents=True, exist_ok=True)

# Add exact price lags required for transparent autoregressive controls.
for months in [2, 3, 6]:
    if f"price_lag_{months}m" not in clean.columns:
        clean = add_exact_calendar_lag(clean, lag_lookup, months=months)

# Create exact 1-, 2-, and 3-month future targets for RQ1 and RQ2.
current_lookup = clean[
    ["region", "Commodity", "date", "price_cleaned_rs_per_quintal"]
].copy()
for horizon in [1, 2, 3]:
    future = current_lookup.copy()
    future["date"] = future["date"] - pd.DateOffset(months=horizon)
    future = future.rename(columns={
        "price_cleaned_rs_per_quintal": f"future_price_h{horizon}m"
    })
    clean = clean.merge(
        future,
        on=["region", "Commodity", "date"],
        how="left",
        validate="one_to_one",
    )
    clean[f"future_price_change_h{horizon}m_pct"] = (
        clean[f"future_price_h{horizon}m"]
        / clean["price_cleaned_rs_per_quintal"]
        - 1
    ) * 100
    clean[f"has_exact_h{horizon}m_target"] = clean[
        f"future_price_h{horizon}m"
    ].notna()

# National future targets remain supplementary and use the same exact-date rule.
national_lookup = national_clean[["date", "food_cpi_2015_100"]].copy()
for horizon in [1, 2, 3]:
    future = national_lookup.copy()
    future["date"] = future["date"] - pd.DateOffset(months=horizon)
    future = future.rename(columns={
        "food_cpi_2015_100": f"future_food_cpi_h{horizon}m"
    })
    national_clean = national_clean.merge(
        future, on="date", how="left", validate="one_to_one"
    )
    national_clean[f"future_food_cpi_change_h{horizon}m_pct"] = (
        national_clean[f"future_food_cpi_h{horizon}m"]
        / national_clean["food_cpi_2015_100"]
        - 1
    ) * 100


def normalize_label(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.casefold()
    )

def normalize_commodity(series):
    normalized = normalize_label(series)
    aliases = {
        "arhar/tur": "arhar (tur/red gram)(whole)",
        "gram": "bengal gram(gram)(whole)",
        "soybean": "soyabean",
    }
    return normalized.replace(aliases)


integration_rows = []

# Genuine arrivals: integrate only the explicitly named standardized file.
arrivals_path = EXTERNAL_REQUIRED / "agmarknet_enam_arrivals.csv.gz"
if arrivals_path.exists():
    arrivals = pd.read_csv(arrivals_path, low_memory=False)
    required = {"date", "state", "commodity", "arrival_quantity"}
    missing = required.difference(arrivals.columns)
    if missing:
        raise ValueError(f"Arrivals file missing required columns: {sorted(missing)}")
    arrivals["date"] = pd.to_datetime(arrivals["date"], errors="coerce").dt.to_period("M").dt.to_timestamp()
    arrivals["region_key"] = normalize_label(arrivals["state"])
    arrivals["commodity_key"] = normalize_commodity(arrivals["commodity"])
    arrivals["arrival_quantity"] = pd.to_numeric(
        arrivals["arrival_quantity"], errors="coerce"
    )
    arrivals_monthly = (
        arrivals.dropna(subset=["date", "region_key", "commodity_key", "arrival_quantity"])
        .groupby(["region_key", "commodity_key", "date"], as_index=False)
        .agg(
            market_arrival_quantity=("arrival_quantity", "sum"),
            arrival_source_rows=("arrival_quantity", "size"),
        )
    )
    clean["region_key"] = normalize_label(clean["region"])
    clean["commodity_key"] = normalize_commodity(clean["Commodity"])
    clean = clean.merge(
        arrivals_monthly,
        on=["region_key", "commodity_key", "date"],
        how="left",
        validate="many_to_one",
    )
    integration_rows.append(["market_arrivals", "integrated", len(arrivals), clean["market_arrival_quantity"].notna().mean()])
else:
    clean["market_arrival_quantity"] = np.nan
    clean["arrival_source_rows"] = np.nan
    integration_rows.append(["market_arrivals", "missing", 0, 0.0])

# State crop area/production/yield: join at state-commodity-year resolution.
apy_path = EXTERNAL_REQUIRED / "des_state_crop_area_production_yield.csv.gz"
if apy_path.exists():
    apy = pd.read_csv(apy_path, low_memory=False)
    required = {"state", "crop", "year", "production_tonne", "yield_kg_per_hectare"}
    missing = required.difference(apy.columns)
    if missing:
        raise ValueError(f"DES APY file missing required columns: {sorted(missing)}")
    apy["region_key"] = normalize_label(apy["state"])
    apy["commodity_key"] = normalize_commodity(apy["crop"])
    apy["analysis_year"] = pd.to_numeric(apy["year"], errors="coerce").astype("Int64")
    apy_year = (
        apy.groupby(["region_key", "commodity_key", "analysis_year"], as_index=False)
        .agg(
            production_tonne=("production_tonne", "sum"),
            yield_kg_per_hectare=("yield_kg_per_hectare", "mean"),
        )
    )
    if "region_key" not in clean:
        clean["region_key"] = normalize_label(clean["region"])
        clean["commodity_key"] = normalize_commodity(clean["Commodity"])
    clean["analysis_year"] = clean["date"].dt.year.astype("Int64")
    clean = clean.merge(
        apy_year,
        on=["region_key", "commodity_key", "analysis_year"],
        how="left",
        validate="many_to_one",
    )
    integration_rows.append(["state_crop_apy", "integrated", len(apy), clean["production_tonne"].notna().mean()])
else:
    clean["production_tonne"] = np.nan
    clean["yield_kg_per_hectare"] = np.nan
    integration_rows.append(["state_crop_apy", "missing", 0, 0.0])

# Rural wages: aggregate occupations to state-month purchasing-power growth.
wage_path = EXTERNAL_REQUIRED / "labour_bureau_rural_wages_monthly.csv.gz"
if wage_path.exists():
    wages = pd.read_csv(wage_path, low_memory=False)
    required = {"state"}
    missing = required.difference(wages.columns)
    if missing:
        raise ValueError(f"Wage file missing required columns: {sorted(missing)}")
    wage_columns = [
        column for column in
        ["male_wage_rs_per_day", "female_wage_rs_per_day"]
        if column in wages.columns
    ]
    if not wage_columns:
        raise ValueError("Wage file requires a male and/or female daily-wage column.")
    for column in wage_columns:
        wages[column] = pd.to_numeric(wages[column], errors="coerce")
    wages["rural_wage_rs_per_day"] = wages[wage_columns].mean(axis=1)
    if "date" in wages.columns:
        wages["date"] = pd.to_datetime(wages["date"], errors="coerce")
    else:
        if not {"year", "month"}.issubset(wages.columns):
            raise ValueError("Wage file requires either date or year/month columns.")
        month_number = pd.to_numeric(wages["month"], errors="coerce")
        if month_number.isna().any():
            month_number = pd.to_datetime(
                wages["month"].astype(str), format="%b", errors="coerce"
            ).dt.month
        wages["date"] = pd.to_datetime(dict(
            year=pd.to_numeric(wages["year"], errors="coerce"),
            month=month_number,
            day=1,
        ), errors="coerce")
    wages["region_key"] = normalize_label(wages["state"])
    wage_monthly = (
        wages.dropna(subset=["date", "region_key", "rural_wage_rs_per_day"])
        .groupby(["region_key", "date"], as_index=False)
        .agg(rural_wage_rs_per_day=("rural_wage_rs_per_day", "mean"))
    )
    if "region_key" not in clean:
        clean["region_key"] = normalize_label(clean["region"])
    clean = clean.merge(
        wage_monthly, on=["region_key", "date"], how="left", validate="many_to_one"
    )
    clean["rural_wage_yoy_pct"] = (
        clean.sort_values(["region", "Commodity", "date"])
        .groupby(["region", "Commodity"], observed=True)["rural_wage_rs_per_day"]
        .pct_change(12, fill_method=None)
        * 100
    )
    wage_status = (
        "integrated_partial_coverage"
        if "coverage_status" in wages.columns
        else "integrated"
    )
    integration_rows.append(["rural_wages", wage_status, len(wages), clean["rural_wage_rs_per_day"].notna().mean()])
else:
    clean["rural_wage_rs_per_day"] = np.nan
    clean["rural_wage_yoy_pct"] = np.nan
    integration_rows.append(["rural_wages", "missing", 0, 0.0])

hces_candidates = list(EXTERNAL_REQUIRED.glob("hces*segment*aggregate*.csv*"))
hces_status = "available_aggregate_for_notebook_08" if hces_candidates else "missing"
integration_rows.append(["hces_microdata", hces_status, 0, float(bool(hces_candidates))])

integration_audit = pd.DataFrame(
    integration_rows,
    columns=["source", "status", "input_rows", "state_panel_match_rate"],
)
integration_audit.to_csv(REPORT_OUTPUT / "02_confirmatory_source_integration.csv", index=False)

target_coverage = pd.DataFrame([
    {
        "horizon_months": horizon,
        "rows_with_target": int(clean[f"has_exact_h{horizon}m_target"].sum()),
        "coverage_rate": float(clean[f"has_exact_h{horizon}m_target"].mean()),
        "target_column": f"future_price_change_h{horizon}m_pct",
    }
    for horizon in [1, 2, 3]
])
target_coverage.to_csv(REPORT_OUTPUT / "02_future_target_coverage.csv", index=False)

print("Exact future-target coverage")
print(target_coverage.to_string(index=False))
print("\nConfirmatory source integration")
print(integration_audit.to_string(index=False))

# Remove temporary normalized keys; the human-readable controlled labels remain.
clean = clean.drop(
    columns=["region_key", "commodity_key", "analysis_year"],
    errors="ignore",
)


Exact future-target coverage
 horizon_months  rows_with_target  coverage_rate               target_column
              1             40189       0.961643 future_price_change_h1m_pct
              2             39587       0.947239 future_price_change_h2m_pct
              3             39162       0.937069 future_price_change_h3m_pct

Confirmatory source integration
         source                              status  input_rows  state_panel_match_rate
market_arrivals                             missing           0                0.000000
 state_crop_apy                          integrated        5397                0.712242
    rural_wages         integrated_partial_coverage        5000                0.018137
 hces_microdata available_aggregate_for_notebook_08           0                1.000000


## 10. Save cleaned datasets and verify round-trip integrity

Compressed CSV keeps files portable in Colab and GitHub while substantially
reducing storage. Each output is read back immediately to verify its row count.


In [11]:
STATE_OUTPUT = PROCESSED / "cleaned_state_monthly.csv.gz"
NATIONAL_OUTPUT = PROCESSED / "cleaned_national_monthly.csv.gz"

clean.to_csv(STATE_OUTPUT, index=False, compression="gzip")
national_clean.to_csv(NATIONAL_OUTPUT, index=False, compression="gzip")

# Read only the key columns during the round-trip check to minimize I/O.
state_roundtrip = pd.read_csv(
    STATE_OUTPUT, usecols=["region", "date", "Commodity"]
)
national_roundtrip = pd.read_csv(NATIONAL_OUTPUT, usecols=["date"])

roundtrip_checks = {
    "state_rows_preserved": len(state_roundtrip) == len(clean),
    "national_rows_preserved": len(national_roundtrip) == len(national_clean),
    "state_keys_still_unique": not state_roundtrip.duplicated(
        ["region", "date", "Commodity"]
    ).any(),
    "national_dates_still_unique": not national_roundtrip.duplicated(["date"]).any(),
}
for check, passed in roundtrip_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {check}")
if not all(roundtrip_checks.values()):
    raise AssertionError("Saved-file round-trip validation failed.")


PASS — state_rows_preserved
PASS — national_rows_preserved
PASS — state_keys_still_unique
PASS — national_dates_still_unique


## 11. Final execution summary

Share the printed JSON after running all cells. Notebook 03 will use these exact
counts and cleaned files for exploratory analysis and visualization.


In [12]:
summary = {
    "notebook": "02_data_quality_and_cleaning_synopsis_aligned",
    "status": "completed",
    "original_research_questions_preserved": True,
    "input_state_rows": int(len(state)),
    "rows_removed_by_validity_rules": int(validity_flags["any_invalid"].sum()),
    "clean_state_rows": int(len(clean)),
    "retention_rate": float(len(clean) / len(state)),
    "extreme_price_rows_flagged": int(clean["price_extreme_flag"].sum()),
    "extreme_price_percent": float(clean["price_extreme_flag"].mean() * 100),
    "rows_with_exact_1m_price_lag": int(clean["has_exact_1m_lag"].sum()),
    "rows_with_exact_12m_price_lag": int(clean["has_exact_12m_lag"].sum()),
    "future_target_rows": {
        f"{horizon}_month": int(clean[f"has_exact_h{horizon}m_target"].sum())
        for horizon in [1, 2, 3]
    },
    "future_target_coverage": {
        f"{horizon}_month": float(clean[f"has_exact_h{horizon}m_target"].mean())
        for horizon in [1, 2, 3]
    },
    "confirmatory_source_status": dict(
        zip(integration_audit["source"], integration_audit["status"])
    ),
    "market_arrivals_is_genuine_quantity_only": True,
    "reporting_counts_not_used_as_arrivals": True,
    "climate_anomaly_rows": int(clean["rainfall_anomaly_pct"].notna().sum()),
    "clean_national_rows": int(len(national_clean)),
    "national_rows_with_cpi": int(national_clean["food_cpi_2015_100"].notna().sum()),
    "state_start": str(clean["date"].min().date()),
    "state_end": str(clean["date"].max().date()),
    "national_start": str(national_clean["date"].min().date()),
    "national_end": str(national_clean["date"].max().date()),
    "missing_prices_interpolated": False,
    "outlier_raw_values_preserved": True,
    "calendar_lag_and_target_method": "exact keyed date joins",
    "output_root": str(OUTPUT_ROOT),
}

SUMMARY_OUTPUT = REPORT_OUTPUT / "02_execution_summary.json"
SUMMARY_OUTPUT.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("=" * 78)
print("NOTEBOOK 02 ALIGNED EXECUTION SUMMARY - PLEASE SHARE THIS OUTPUT")
print("=" * 78)
print(json.dumps(summary, indent=2))
print("\nGenerated files:")
for path in [
    STATE_OUTPUT,
    NATIONAL_OUTPUT,
    REPORT_OUTPUT / "02_state_missingness.csv",
    REPORT_OUTPUT / "02_national_missingness.csv",
    REPORT_OUTPUT / "02_commodity_coverage.csv",
    REPORT_OUTPUT / "02_validity_rule_counts.csv",
    REPORT_OUTPUT / "02_extreme_price_summary.csv",
    REPORT_OUTPUT / "02_confirmatory_source_integration.csv",
    REPORT_OUTPUT / "02_future_target_coverage.csv",
    SUMMARY_OUTPUT,
]:
    print(f"- {path} ({path.stat().st_size / 1_000_000:.3f} MB)")


NOTEBOOK 02 ALIGNED EXECUTION SUMMARY - PLEASE SHARE THIS OUTPUT
{
  "notebook": "02_data_quality_and_cleaning_synopsis_aligned",
  "status": "completed",
  "original_research_questions_preserved": true,
  "input_state_rows": 41792,
  "rows_removed_by_validity_rules": 0,
  "clean_state_rows": 41792,
  "retention_rate": 1.0,
  "extreme_price_rows_flagged": 15,
  "extreme_price_percent": 0.035892036753445634,
  "rows_with_exact_1m_price_lag": 40189,
  "rows_with_exact_12m_price_lag": 37090,
  "future_target_rows": {
    "1_month": 40189,
    "2_month": 39587,
    "3_month": 39162
  },
  "future_target_coverage": {
    "1_month": 0.9616433767228177,
    "2_month": 0.9472387059724349,
    "3_month": 0.9370692955589587
  },
  "confirmatory_source_status": {
    "market_arrivals": "missing",
    "state_crop_apy": "integrated",
    "rural_wages": "integrated_partial_coverage",
    "hces_microdata": "available_aggregate_for_notebook_08"
  },
  "market_arrivals_is_genuine_quantity_only": true,


## Notebook 02 conclusion

The cleaned panel now contains exact one-, two-, and three-month future
commodity-price targets required by the original RQ1 and RQ2. Missing
confirmatory sources are visible in the integration audit and remain missing
rather than being replaced with fabricated values. Notebook 03 may proceed for
interim EDA while the official downloads continue in parallel.
